<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import json
import pennylane as qml
import pennylane.numpy as np

def qsp_matrix(a, angles):
    """Constructs the QSP matrix for a given a and phase angles using qml.qsvt."""
    dev = qml.device("default.qubit", wires=1)
    
    @qml.qnode(dev)
    def circuit():
        qml.qsvt(a, angles, wires=0)
        return qml.state()
    
    # Execute the circuit to get the state vector
    state = circuit()
    # From the state vector, construct the matrix (since we started in |0>)
    mat = np.column_stack([state, np.conj(state)])[:, 0]
    
    return mat

def coefficients(angles):
    """This function returns the coefficients associated with the polynomial generated by 
    the QSP routine as a function of the phase angles.
    
    Args:
        - angles (np.array(float)): Array of real numbers containing the four phase angles,
        in reverse order of application.
    
    Returns:       
        - (np.array(complex)): A numpy array containing the coefficients of the polynomial
        generated by QSP, where the first element is the coefficient of the cubic term and
        the second element is for the linear term.
    """
    # We'll sample the polynomial at specific values of 'a' to reconstruct the coefficients
    a_values = np.linspace(-1, 1, 10)  # Sample points for 'a'
    
    # Store the values of the polynomial P(a)
    p_values = []
    
    for a in a_values:
        # Compute the QSP matrix for this 'a'
        p = qsp_matrix(a, angles)
        # The (0,0) element of the matrix is P(a)
        p_values.append(p[0])
    
    # Construct a Vandermonde matrix for the polynomial fitting
    A = np.vstack([a_values**3, a_values]).T
    
    # Solve the linear system Ac = p to find the coefficients
    # We use the pseudoinverse for a least-squares solution
    A_pinv = np.linalg.pinv(A)
    coeffs = A_pinv @ np.array(p_values)
    
    # Return the coefficients [alpha, beta] for a^3 and a terms
    return coeffs[[0, 1]]

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    outs = [[float(np.real(elem)), float(np.imag(elem))] for elem in coeffs]
    
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol = 1e-4)

# These are the public test cases
test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


SyntaxError: invalid syntax (1351478239.py, line 1)